## Import 

In [0]:
from pyspark.sql.functions import lit

##Bronze Layer ?

In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.gold')

## Params & Inputs 

In [0]:
dbutils.widgets.text("city", "lyon")
dbutils.widgets.text("table_bronze", "neighbourhoods")
dbutils.widgets.text("file", "neighbourhoods.geojson")
user_email = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
dbutils.widgets.text("path_file", f"/Workspace/Users/{user_email}/raw/","full path avec / /")

city = dbutils.widgets.get("city")
table_bronze = dbutils.widgets.get("table_bronze")
file = dbutils.widgets.get("file")
path_file = dbutils.widgets.get("path_file")


## read

In [0]:

df = (
    spark.read
    .format("json")
    .option("multiline", "true")  
    .load(f"{path_file}{city}/{file}")
    .withColumn("city", lit(city))
)

## Save

In [0]:
df.write.format("delta").mode("overwrite").option("replaceWhere",f"city = '{city}'").saveAsTable(f"bronze.{table_bronze}")